In [232]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 12.df 데이터셋에서 “상권업종대분류명”이 음식인 데이터 중 서울특별시 데이터 서브셋
- https://seaborn.pydata.org/tutorial/categorical.html : 범주형그래프
## ①	“상권업종대분류명”이 음식인 서브셋 중 서울특별시 데이터만 변수 df_seoul_food에 할당하고 확인


In [35]:
# 시리즈 끼리의 and연산 : & 
df_seoul_food = df[(df['시도명']=='서울특별시') & (df['상권업종대분류명']=='음식')]
df_seoul_food.shape

(138558, 17)

In [38]:
df_seoul_food = df_food[df_food['시도명']=='서울특별시']
df_seoul_food.shape

(138558, 17)

## ② df_seoul_food 데이터 셋을 시군구명, 상권업종중분류명으로 그룹화하여 상점수를 count한 내용을 food_gu 변수에 할당. 

In [50]:
food_gu = df_seoul_food.groupby(['시군구명', '상권업종중분류명'])['상호명'].count()
food_gu

시군구명  상권업종중분류명
강남구   구내식당·뷔페      128
      기타 간이       1956
      기타 외국          2
      동남아시아        174
      비알코올        2093
                  ... 
중랑구   서양식           87
      일식           137
      주점           571
      중식           106
      한식          1879
Name: 상호명, Length: 239, dtype: int64

## ③ food_gu 변수를 표로 출력(food_gu 이용하여 unstack).

In [ ]:
# unstack() : 0 level의 index

food_gu.unstack().replace(np.nan, '')

## ④ 위 3번 스타일의 표를 pivot_table함수를 이용하여 출력

In [ ]:
df_seoul_food.pivot_table(index='시군구명', 
                          columns='상권업종중분류명', 
                          values='상호명', 
                          aggfunc='count').replace(np.nan, '')

## ⑤ 3번의 결과 중 강남구 데이터만 뽑아 barplot으로 시각화(판다스 plot이용)

In [ ]:
r = food_gu['강남구'].sort_values(ascending=False)
r.plot(kind='bar', rot=0)
for i, v in enumerate(r):
    plt.text(i, v, v, ha='center')
plt.show()

## ⑥ 3번 food_gu를 seaborn을 이용하여 구별 음식점 상호 개수를 시각화

In [ ]:
food_gu_resetindex = food_gu.reset_index().rename(columns={'상호명':'상호수'})
food_gu_resetindex

In [ ]:
plt.rcParams['figure.figsize']=(17, 3)
sns.barplot(data=food_gu_resetindex,
           x='시군구명',
           y='상호수',
           errorbar=None,
           order=food_gu_resetindex.groupby('시군구명')['상호수'].sum().sort_values().index)
           # estimator='mean'
plt.xticks(rotation=30)
plt.show()

In [ ]:
# 구별, 상권업종중분류별 batplot
sns.barplot(data=food_gu_resetindex,
           x='시군구명',
           y='상호수',
           hue='상권업종중분류명')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

In [ ]:
# 구별, 상권업종중분류별 catplot(서브플롯)
import time
start = time.time() # 1970.1.1부터 이 시점까지의 초수
sns.catplot(data=food_gu_resetindex.sort_values('상호수', ascending=False),
           x='시군구명',
           y='상호수',
           kind='bar',
           col='상권업종중분류명', col_wrap=1,
           hue='시군구명', dodge=False, # 막대를 겹치지 않고 원래 너비로 'dodge'
           sharey=False, sharex=False,
           height=2, aspect=8, # 높이, (높이 대비 가로 사이즈 비율)
           )

plt.show()
print(f'그래프 그리는데 걸린 시간 : {time.time()-start:.0f}초')

## ⑦ 상권업종중분류명별 음식점 상호갯수

In [ ]:
food_gu_resetindex.groupby('상권업종중분류명')['상호수'].sum() #정렬전
df_seoul_food['상권업종중분류명'].value_counts() # 정렬 포함

In [ ]:
# 상권업종중분류별 상호수(음식점 갯수)
sns.barplot(data=food_gu_resetindex,
           x='상권업종중분류명',
           y='상호수',
           estimator='sum',
           errorbar=None, 
           order=df_seoul_food['상권업종중분류명'].value_counts().index)

plt.title('상권업종중분류별 상호수 갯수')
plt.show()

In [ ]:
# 상권업종중분류별 상호수(음식 갯수)
sns.countplot(data=df_seoul_food,
             x='상권업종중분류명',
             order=df_seoul_food['상권업종중분류명'].value_counts().index)
plt.show()

## ⑧	Seaborn의 catplot을 이용하여 상권업종중분류별 음식점을 구별로 시각화(서브플롯으로 시각화)

https://seaborn.pydata.org/tutorial/categorical.html

In [ ]:
# 구별, 상권업종중분류별
sns.barplot(data=food_gu_resetindex,
           x='상권업종중분류명',
           y='상호수',
           hue='시군구명')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

plt.show()

In [ ]:
# 구별 서브플롯
import time
start = time.time()
g = sns.catplot(data=food_gu_resetindex.sort_values('상호수'),
           x='상권업종중분류명',
           y='상호수',
#           hue='상권업종중분류명', dodge=False,
           color='orange',
           col='시군구명',
           kind='bar',
           col_wrap=2, estimator=sum,
           sharex=False, sharey=False,
           height=2,
           aspect=4)
# for ax in g.axes.flat:
#     ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontweight='bold')
g.set_xticklabels(rotation=45, fontweight='bold')
g.tight_layout()
plt.show()
print(f'그래프 그린 시간 : {time.time()-start}초')

## ⑨	Seaborn의 catplot을 이용하여 구별 음식점을 상권업종중분류명별로 서브 플롯으로 시각화

In [ ]:
# 구별 서브플롯
import time
start = time.time()
g = sns.catplot(data=food_gu_resetindex.sort_values('상호수'),
           x='상권업종중분류명',
           y='상호수',
           color='orange', # hue='상권업종중분류명', dodge=False,
           col='시군구명',
           kind='bar',
           col_wrap=2, estimator=sum,
           sharex=False, sharey=False, # 축 공유 X
           height=2, # 높이
           aspect=4 ) # 높이 대비 가로 비율
# for ax in g.axes.flat:
#     ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontweight='bold')
g.set_xticklabels(rotation=45, fontweight='bold') # x축 커스토마이징
g.tight_layout() # 서브플롯끼리 겹치지 않게
plt.show()
print(f'그래프 그린 시간 : {time.time()-start}초')

# 13.	구별로 학원수 비교 : 서울 대치동이나 목동에 사교육이 발달되었다는 가설을 뒷받침할 수 있는 분석
## ① 서울시 교육(상권업종대분류명 이용) 데이터를 df_academy 변수에 할당하고 확인


In [86]:
df['상권업종대분류명'].unique()

array(['부동산', '음식', '시설관리·임대', '예술·스포츠', '소매', '과학·기술', '수리·개인', '보건의료',
       '숙박', '교육'], dtype=object)

In [96]:
df_academy = df[(df['상권업종대분류명']=='교육') & (df['시도명']=='서울특별시')].copy(
                                                                        # deep=True
                                                                   )
df_academy.shape
# df_academy를 수정할 경우 .copy() 깊은 복사 함수로 추출 warning 출력 안 됨

(45080, 17)

In [97]:
df_academy.to_csv('c:/ai/downloads/shareData/상가정보/ch13_df_academy.csv', index=False)

## ②	df_academy 데이터 셋을 상호명별로 빈도수 출력(value_counts()함수 이용하거나 groupby이용)

In [ ]:
df_academy['상호명'].value_counts()

## ③ df_academy 데이터 셋을 상호명별로 빈도수 상위 10개 출력

In [ ]:
df_academy['상호명'].value_counts().head(10)
df_academy['상호명'].value_counts()[:10]
df_academy['상호명'].value_counts().iloc[:10]

## ④ df_academy 데이터 셋을 시군구명 별로 빈도수 출력(학원이 가장 많은 구부터 출력)

In [ ]:
# df_academy.groupby('시군구명')['상호명'].count()
df_academy['시군구명'].value_counts()

In [ ]:
df_academy['시군구명'].value_counts().plot(kind='bar', rot=0)
plt.show()

In [ ]:
sns.countplot(data=df_academy,
             x='시군구명',
             order=df_academy['시군구명'].value_counts().index)
plt.show()

## ⑤df_academy 데이터 셋에서 어떤 종류의 학원들이 많은지 상위10개만 academy_count변수에 할당하고 출력(상권업종소분류명 컬럼 이용)

In [103]:
group = df_academy.groupby(['상권업종중분류명','상권업종소분류명'])
for (medium, sub), val in group:
    print(medium, ' => ',sub)

교육 지원  =>  교육컨설팅업
교육 지원  =>  기타 교육지원 서비스업
기타 교육  =>  그 외 기타 교육기관
기타 교육  =>  기타 기술/직업 훈련학원
기타 교육  =>  기타 예술/스포츠 교육기관
기타 교육  =>  레크리에이션 교육기관
기타 교육  =>  미술학원
기타 교육  =>  사회교육시설
기타 교육  =>  외국어학원
기타 교육  =>  요가/필라테스 학원
기타 교육  =>  운전학원
기타 교육  =>  음악학원
기타 교육  =>  전문자격/고시학원
기타 교육  =>  직원 훈련기관
기타 교육  =>  청소년 수련시설
기타 교육  =>  컴퓨터 학원
기타 교육  =>  태권도/무술학원
일반 교육  =>  입시·교과학원


In [104]:
df_academy = pd.read_csv("c:/ai/downloads/shareData/상가정보/ch13_df_academy.csv")
df_academy.shape

(45080, 17)

In [106]:
# 어떤 종류의 학원이 있는(= 어떤 상권업종소분류명)
print(df_academy['상권업종소분류명'].unique())
print(len(df_academy['상권업종소분류명'].unique()))
print(df_academy['상권업종소분류명'].nuniqueque())

['입시·교과학원' '음악학원' '태권도/무술학원' '요가/필라테스 학원' '미술학원' '교육컨설팅업' '전문자격/고시학원'
 '기타 기술/직업 훈련학원' '청소년 수련시설' '사회교육시설' '직원 훈련기관' '기타 예술/스포츠 교육기관'
 '기타 교육지원 서비스업' '그 외 기타 교육기관' '외국어학원' '운전학원' '레크리에이션 교육기관' '컴퓨터 학원']
18
18


In [ ]:
# 상권업종소분류명별 빈도수(빈도수가 상위 10개)
df_academy['상권업종소분류명'].value_counts().head(10)
df_academy['상권업종소분류명'].value_counts().iloc[:10]
academy_count = df_academy['상권업종소분류명'].value_counts()[:10]
academy_count

## ⑥ df_academy 데이터셋에서 상권업종소분류명별로 빈도수를 구했을 때 빈도가 1000이상인 데이터만 따로 academy_count_1000변수에 할당

In [ ]:
academy_count = df_academy['상권업종소분류명'].value_counts()
academy_count_1000 = academy_count[academy_count>1000]
academy_count_1000

In [116]:
# 소분류가 academey_count_1000인 행만 추출
top1000 = df_academy[df_academy['상권업종소분류명'].isin(academy_count_1000.index)]
top1000.shape

(42492, 17)

In [ ]:
# 구별 학원(top1000) 수 비교
top1000['시군구명'].value_counts()
top1000['시군구명'].value_counts().plot(kind='bar')
plt

## ⑦ df_academy 데이터셋을 “시군구명”, "상권업종소분류명” 별 상호명 빈도수를 academy_group 변수에 할당 출력

In [ ]:
academy_group = df_academy.groupby(['시군구명', '상권업종소분류명'])['상호명'].count()
academy_group

In [ ]:
academy_group['중구']

## ⑧ academy_group 데이터셋에서 강남구 데이터만 출력 및 시각화(barplot)

In [ ]:
result = academy_group['강남구'].sort_values(ascending=False).head(10)
result.plot(kind='bar', rot=45)
for i, v in enumerate(result):
    plt.text(i, v, v, ha='center')
plt.grid(axis='y')
plt.show()

In [ ]:
sns.barplot(x=result.index, y=result)
plt.xticks(rotation=45)
for i, v in enumerate(result):
    plt.text(i, v, v, ha='center')
plt.grid(axis='y')
plt.show()

## ⑨ df_academy데이터 중 “법정동명”컬럼이 “대치동”과 “목동”인 데이터만 가져와 상권업종소분류명별 빈도수 출력

In [125]:
# 대치동에 많은 학원 종률
df_academy.loc[df_academy['법정동명']=='대치동', '상권업종소분류명'].value_counts()[:3]

입시·교과학원    1268
교육컨설팅업      130
미술학원         93
Name: 상권업종소분류명, dtype: int64

In [141]:
# 목동에 많은 학원 종류
df_academy.loc[df_academy['법정동명']=='목동', '상권업종소분류명'].value_counts()[:3]

입시·교과학원       711
음악학원           87
요가/필라테스 학원     83
Name: 상권업종소분류명, dtype: int64

In [218]:
# 대치동과 목동에 많은 학원 종류
top2 = df_academy.loc[df_academy['법정동명'].isin(['대치동','목동']), 
                      '상권업종소분류명'].value_counts().head(3)
top2 = top2.drop('요가/필라테스 학원')
top2

입시·교과학원    1979
교육컨설팅업      158
Name: 상권업종소분류명, dtype: int64

In [219]:
# df_academy에서 top2.index 소분류명만 추출
df_academy_sel = df_academy[df_academy['상권업종소분류명'].isin(top2.index)]
df_academy_sel.shape

(18907, 17)

In [ ]:
# 입시관련 학원이 많은 법정동명
df_academy_sel['법정동명'].value_counts().head(10)

## ⑩“상권업종소분류명”별 "시군구명” 별 상호명 빈도수를 g변수에 할당하고 출력

In [ ]:
g = df_academy.groupby(['상권업종소분류명','시군구명'])['상호명'].count()
g

In [ ]:
g['태권도/무술학원'].sort_values(ascending=False)

## ⑪ g변수의 내용중 "상권업종소분류명” 컬럼이 “입시·교과학원”데이터만 시각화(pandas의 plot.bar, pandas의 barh, seaborn의 barplot)

In [ ]:
result = g['입시·교과학원'].sort_values(ascending=False)
result

In [ ]:
plt.figure(figsize=(18,3))
result.plot(kind='bar', rot=0)
plt.show()

In [ ]:
plt.figure(figsize=(18,3))
sns.barplot(x=result.index, y=result)
plt.xticks(rotation=10)
plt.show()

# 14. 서울시 데이터만 경도와 위도를 산점도로 시각화
## ① df_academy 데이터셋의 경도와 위도를 “시군명”별로 색상을 다르게 scatterplot으로 시각화


In [ ]:
sns.scatterplot(data=df_academy,
               x='경도',
               y='위도',
               hue='상권업종소분류명', palette='Set1')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='외국어학원'],
               x='경도',
               y='위도',
               hue='상권업종소분류명', palette='Set1')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

https://stackoverflow.com/questions/30490740/move-legend-outside-figure-in-seaborn-tsplot : 범례사용

## ② df_academy 데이터셋의 경도와 위도를 “상권업종소분류명”별로 색상을 다르게 scatterplot으로 시각화

In [ ]:
sns.scatterplot(data=df_academy,
               x='경도',
               y='위도',
               hue='상권업종소분류명', palette='Set1')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='외국어학원'],
               x='경도',
               y='위도',
               hue='상권업종소분류명', palette='Set1')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
df_academy_sel['상권업종소분류명'].unique()

In [ ]:
custom_palette = {'입시·교과학원':'y',
                  '교육컨설팅업':'#00ff00'}
sns.scatterplot(data=df_academy_sel,
               x="경도", y="위도", 
               hue='상권업종소분류명',
               palette=custom_palette)
plt.show()

## ③ df_academy 데이터셋 중 “입시·교과학원” 데이터만, 경도와 위도를 “시군구명”별로 색상을 다르게 scatterplot으로 시각화

In [ ]:
sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='입시·교과학원'],
               x='경도',
               y='위도',
               hue='시군구명')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## ④ df_academy 데이터셋 중 “태권도/무술학원” 데이터만, 경도와 위도를 “시군명”별로 색상을 다르게 scatterplot으로 시각화

In [ ]:
sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='태권도/무술학원'],
               x='경도',
               y='위도',
               hue='시군구명')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## ⑤ df_academy 데이터셋 중 “입시·교과학원” 데이터와 “태권도/무술학원” 데이터만, 경도와 위도를 “상권업종소분류명”별로 색상을 다르게 scatterplot으로 시각화

In [156]:
df_academy[df_academy['상권업종소분류명'].isin(['입시·교과학원', '태권도/무술학원'])].shape

(19350, 17)

In [ ]:
sns.scatterplot(data=df_academy[df_academy['상권업종소분류명'].isin(['입시·교과학원', '태권도/무술학원'])],
               x='경도', y='위도',
               hue='상권업종소분류명',
               palette=['darkorange', 'b'])
plt.show()

# 15. 지도시각화 : Folium
` 아나콘다 프롬프트에서 아래의 둘 중 하나를 실행`

`pip install folium`

`conda install -c conda-forge folium`

-	docs : https://python-visualization.github.io/folium/latest/getting_started.html?utm_source=chatgpt.com
-	Quickstart : https://python-visualization.github.io/folium/version-v0.9.1/quickstart.html?utm_source=chatgpt.com
- API reference : https://python-visualization.github.io/folium/latest/reference.html

In [160]:
import folium
folium.__version__

'0.20.0'

In [ ]:
m = folium.Map(location=[45.5236, -122.6750],
              zoom_start=12,
              #width='90%',
              width=800,
              #height=400
               height='50%'
              )
m

In [167]:
# 상권 업종소분류명이 태권도/무술학원과 입시·교과학원인데이터만
part = ['입시·교과학원', '태권도/무술학원']
df_m = df_academy[df_academy['상권업종소분류명'].isin(part)].sample(1000, random_state=3)
df_m.shape

(1000, 17)

In [168]:
df_m.head().index

Int64Index([42228, 23228, 42006, 39801, 34219], dtype='int64')

In [ ]:
# df_m을 지도시각화
lat_mean = df_m['위도'].mean()
long_mean = df_m['경도'].mean()
m = folium.Map(location=[lat_mean, long_mean],
              zoom_start=15)
m

- 지도에 marker 추가하기

In [ ]:
m = folium.Map([lat_mean, long_mean], zoom_start=12)

folium.Marker(
    location=[37.527806, 127.128551],
    tooltip="리딩오션강동독서논술교습소 - 강동구 성내로15길",
    icon=folium.Icon(icon="cloud"),
).add_to(m)

lat = df_m.loc[23228, '위도']
long = df_m.loc[23228, '경도']
tooltip = "<b>{} - {}</b>".format(df_m.loc[23228, '상호명'],
                                 df_m.loc[23228, '도로명'][6:])
folium.Marker(
    location=[lat, long],
    tooltip=tooltip, # 마우스 up시 나타나는 글씨
    # popup="Timberline Lodge", click후 나타나는 글씨
    icon=folium.Icon(color="green"),
).add_to(m)

m

In [181]:
for i in df_m[:10].index:
    print(df_m.loc[i, '상호명'], df_m.loc[i, '도로명'][6:], df_m.loc[i, '위도'], df_m.loc[i, '경도'])

리딩오션강동독서논술교습소 강동구 성내로15길 37.5278059885402 127.128551392731
마운틴영어교습소 동작구 사당로23나길 37.4876976095013 126.97554272704
피오르에듀 강남구 도곡로77길 37.4999485460077 127.058128022903
김태리 마포구 고산2길 37.5536648722227 126.939080234854
임팩트학원 광진구 자양로 37.547943768781 127.088945333273
정릉백상영수학원 성북구 정릉로26길 37.6040365796062 127.01156251141
에폴영어학원 강서구 공항대로55길 37.5544282495679 126.861477730045
성북스마트정일학원 성북구 동소문로 37.5923324776886 127.01368940918
마포서대문와이즈만 마포구 마포대로 37.5423079661221 126.948967775601
본스타컴퍼니 서초구 나루터로 37.5157291555181 127.017374210867


In [ ]:
m = folium.Map([lat_mean, long_mean], 
               zoom_start=12)

for i in df_m.index:
    lat = df_m.loc[i, '위도']
    long = df_m.loc[i, '경도']
    tooltip ='<b>{}-{}</b>'.format(df_m.loc[i, '상호명'],
                                   df_m.loc[i, '도로명'][6:]) 

#     folium.Marker(
#         location=[lat, long],
#         tooltip=tooltip,
#         icon=folium.Icon(icon="cloud"),
#     ).add_to(m)
    folium.Circle(
        radius=100,
        location=[lat, long],
        tooltip=tooltip,
        color='crimson',
        fill=False
    ).add_to(m)
m

In [192]:
# 태권도/무술학원과 입시·교과학원을 분리해서 지도 시각화
df1 = df_m[df_m['상권업종소분류명']=='태권도/무술학원']
df2 = df_m[df_m['상권업종소분류명']=='입시·교과학원']
print(df1.shape)
print(df2.shape)

(126, 17)
(874, 17)


In [195]:
m = folium.Map([lat_mean, long_mean], 
               zoom_start=12)

for i in df1.index:
    lat = df_m.loc[i, '위도']
    long = df_m.loc[i, '경도']
    tooltip ='<b>{}-{}</b>'.format(df_m.loc[i, '상호명'],
                                   df_m.loc[i, '도로명'][6:]) 
    folium.Circle(
        radius=100,
        location=[lat, long],
        tooltip=tooltip,
        color='red',
        fill=False
    ).add_to(m)
    
for i in df2.index:
    lat = df_m.loc[i, '위도']
    long = df_m.loc[i, '경도']
    tooltip ='<b>{}-{}</b>'.format(df_m.loc[i, '상호명'],
                                   df_m.loc[i, '도로명'][6:]) 
    folium.Circle(
        radius=100,
        location=[lat, long],
        tooltip=tooltip,
        color='blue',
        fill=False
    ).add_to(m)
m

In [196]:
m.save('ch13.html')